# Bateria de 30 dias — Random Forest

Versão paralela do notebook oficial de 7 dias (em `horizonte_7dias/`),
**re-treinada do zero** para o horizonte de **30 dias**
(`contamination_level = 30`): rótulos, busca de hiperparâmetros (própria,
não reaproveitada de 7 dias), fit final no treino completo, limiar ótimo na
validação e métricas são todos recalculados. Mantém-se a janela oficial
`WINDOW = 90`, os mesmos splits por disco, o mesmo espaço de busca, o mesmo
`n_iter`/`cv` e `scoring='average_precision'` da bateria de 7 dias.

**Atenção (não comparar entre horizontes):** a prevalência de positivos em
30 dias (~8%) é cerca de 4× a de 7 dias (~2%), o que infla F1/AP
mecanicamente. Compare os modelos *dentro* do horizonte de 30 dias (e use
`AMA_comparacao_modelos_30d.ipynb`), não os números de 30d contra os de 7d.
Todas as saídas levam o sufixo `_30d`.

# Predicao de Falhas em SSDs - Random Forest

Este notebook treina um **Random Forest** para o mesmo problema resolvido em
`AMA_projeto_LSTM.ipynb` (baseline) e em `AMA_projeto_LogisticRegression.ipynb`:
prever, para cada disco e cada dia, se ele **vai falhar dentro dos proximos
`CONTAMINATION_LEVEL` dias**.

Para garantir uma comparacao justa entre os tres modelos do trabalho (LSTM,
Regressao Logistica, Random Forest), este notebook reaproveita - via o modulo
`ssd_utils.py` - exatamente os mesmos componentes usados no notebook de
Regressao Logistica:

- **os mesmos indices de split** treino/teste/validacao do notebook do LSTM
  (`[0:3740]`, `[3740:4541]`, `[4541:]`);
- **a mesma funcao de rotulagem** `create_class_labels` (copia literal da
  celula 8 do notebook do LSTM), com `contamination_level = 30`;
- **as mesmas features de janela deslizante** (`build_windowed_features`):
  para cada `(disco, dia)`, um resumo dos ultimos `WINDOW` dias de leitura
  SMART -> `[atual, media, desvio padrao, minimo, maximo]` por atributo
  (24 x 5 = 120 features). Isso da contexto temporal de curto prazo a um
  modelo que, assim como a Regressao Logistica, nao enxerga a sequencia
  inteira como o LSTM enxerga;
- **as mesmas metricas de avaliacao**: acuracia, precisao, recall, F1-score
  e matriz de confusao, calculadas apenas sobre os timesteps validos
  (`mask == 1`).

### Por que Random Forest?

E um ensemble de arvores de decisao que captura relacoes **nao-lineares** e
interacoes entre atributos sem exigir normalizacao dos dados - ao contrario
da Regressao Logistica. Tambem fornece **importancia de features**, util para
discutir, no artigo, quais atributos SMART sao mais informativos para prever
falha.

### Resultados

Ao final, o notebook salva `resultados_random_forest_30d.pkl` no **mesmo formato**
usado pelo notebook de Regressao Logistica (`resultados_logistic_regression.pkl`),
permitindo montar os graficos comparativos finais (ROC, precisao-recall,
barras de metricas, tempo de treino) entre os tres modelos para a apresentacao.

> **Nota:** este notebook nao e executado agora (ambiente sem GPU/recursos
> limitados). Sera executado depois, em outra maquina, junto com o de
> Regressao Logistica.

In [1]:
# ===== Configuracao =====
import os
import time
import pickle

import numpy as np

import os
import sys

# --- Resolucao robusta de caminhos (independente do diretorio de trabalho) ---
def _encontra_raiz(inicio=None):
    """Sobe na arvore ate achar comum/ssd_utils.py."""
    d = os.path.abspath(inicio or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, 'comum', 'ssd_utils.py')):
            return d
        pai = os.path.dirname(d)
        if pai == d:
            raise RuntimeError('raiz do projeto (com comum/ssd_utils.py) nao encontrada')
        d = pai

PROJ_ROOT = _encontra_raiz()
COMUM_DIR = os.path.join(PROJ_ROOT, 'comum')
H7_DIR = os.path.join(PROJ_ROOT, 'horizonte_7dias')
H30_DIR = os.path.join(PROJ_ROOT, 'horizonte_30dias')
EXP_DIR = os.path.join(PROJ_ROOT, 'experimentos')
if COMUM_DIR not in sys.path:
    sys.path.insert(0, COMUM_DIR)
# ---------------------------------------------------------------------------
import ssd_utils as ssd

DATA_DIR = COMUM_DIR        # data.pickle / mask.pickle (pasta comum/)
OUTPUT_DIR = H30_DIR        # resultados, modelos e figuras (bateria de 30 dias)
WINDOW = 90                 # dias de historico resumidos em cada amostra (atual/media/desvio/min/max); ver AMA_experimento_janela.ipynb
CONTAMINATION_LEVEL = 30     # horizonte de previsao de falha - igual ao notebook do LSTM e do baseline
RANDOM_STATE = 42

## 1. Carregamento dos dados, rotulos e construcao das features

Identico ao notebook de Regressao Logistica - ver aquele notebook para uma
explicacao detalhada de `create_class_labels` e `build_windowed_features`.
Repetimos aqui para que este notebook seja executavel de forma independente.

In [2]:
data, mask = ssd.load_dataset(DATA_DIR)
print(f'data: {data.shape}   mask: {mask.shape}')

labels = ssd.create_class_labels(data, mask, CONTAMINATION_LEVEL)
print(f'labels: {labels.shape}   positivos: {int(labels.sum())} '
      f'({labels.mean()*100:.3f}% dos timesteps - desbalanceamento extremo, '
      f'class_weight="balanced" e essencial)')

start = time.time()
X_all, disco_id, timestep, feature_cols = ssd.build_windowed_features(data, mask, window=WINDOW)
print(f'X_all: {X_all.shape}   ({len(feature_cols)} features por amostra)')
print(f'Tempo de construcao das features: {time.time() - start:.1f}s')

data: (5343, 360, 24)   mask: (5343, 360)
labels: (5343, 360)   positivos: 158845 (8.258% dos timesteps - desbalanceamento extremo, class_weight="balanced" e essencial)


X_all: (1923480, 120)   (120 features por amostra)
Tempo de construcao das features: 21.6s


## 2. Divisao treino / teste / validacao

Mesmos indices de disco do notebook do LSTM e do baseline de Regressao
Logistica. Descartamos o padding (`mask == 0`), assim como o `batch_evaluation`
do LSTM ignora essas posicoes ao calcular metricas e perda.

In [3]:
splits_idx = ssd.split_indices(n_discos=data.shape[0])
y_flat = labels.reshape(-1)
mask_flat = mask.reshape(-1)

train = ssd.select_split(X_all, disco_id, timestep, y_flat, mask_flat, splits_idx['train'])
test = ssd.select_split(X_all, disco_id, timestep, y_flat, mask_flat, splits_idx['test'])
validation = ssd.select_split(X_all, disco_id, timestep, y_flat, mask_flat, splits_idx['validation'])

for nome, split in [('treino', train), ('teste', test), ('validacao', validation)]:
    n = len(split['y'])
    n_pos = int(split['y'].sum())
    print(f'{nome:10s}: {n:>9,} amostras validas | positivas: {n_pos:>6,} ({n_pos/n*100:.3f}%)')

X_train, y_train = train['X'], train['y']
X_test, y_test = test['X'], test['y']
X_validation, y_validation = validation['X'], validation['y']

treino    : 1,240,487 amostras validas | positivas: 111,132 (8.959%)
teste     :   265,245 amostras validas | positivas: 23,845 (8.990%)
validacao :   267,254 amostras validas | positivas: 23,868 (8.931%)


## 3. (Sem padronizacao)

Diferente da Regressao Logistica, **arvores de decisao sao invariantes a
escala** - elas particionam o espaco de features por limiares (`feature <= valor`),
entao normalizar nao muda a estrutura aprendida. Usamos as features brutas
diretamente, o que tambem evita um passo de pre-processamento a mais.

## 4. Busca de hiperparametros

Assim como na Regressao Logistica, `class_weight='balanced'` compensa o forte
desbalanceamento (~0,3-2% de amostras positivas). A busca usa
`RandomizedSearchCV` (mais barato que grid search exaustivo num dataset deste
tamanho) variando profundidade, numero de arvores e criterios de divisao,
otimizando a **Average Precision da classe positiva**.

A busca otimiza **Average Precision** (`scoring='average_precision'`), e
nao F1: a comparacao entre os modelos do trabalho e feita por AP (metrica
de ranking, independente de limiar) e o ponto de operacao e escolhido
depois, na validacao. Selecionar hiperparametros pelo mesmo criterio evita
o vies do corte fixo de 0,5 embutido no F1 do `scoring='f1'`.

A busca roda numa **subamostra estratificada** do treino (ate 300 mil linhas) e o **ajuste final** e feito no treino completo. Alem disso, o paralelismo fica em **um nivel so**: o `RandomForestClassifier` usa todos os nucleos (`n_jobs=-1`) e o `RandomizedSearchCV` serializa os fits (`n_jobs=1`) - evita o *oversubscription* de CPU que trava o Colab (2 nucleos) com paralelismo aninhado.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split

param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
}

base_rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

search = RandomizedSearchCV(
    base_rf,
    param_distributions=param_distributions,
    n_iter=12,
    scoring='average_precision',
    cv=3,
    n_jobs=1,            # serializa os fits do search: o paralelismo fica so no RF (n_jobs=-1),
                         # evitando oversubscription de CPU (trava no Colab de 2 nucleos)
    random_state=RANDOM_STATE,
    verbose=2,
)

# Busca numa subamostra estratificada do treino; o fit final (mais abaixo) e no
# treino completo. Pratica honesta e muito mais leve - essencial no Colab (2 nucleos).
SUBSAMPLE_SIZE = min(300_000, len(X_train) - 2)
X_search, _, y_search, _ = train_test_split(
    X_train, y_train, train_size=SUBSAMPLE_SIZE,
    stratify=y_train, random_state=RANDOM_STATE,
)
print(f'Busca em subamostra: {X_search.shape[0]:,} de {len(X_train):,} amostras de treino')

start = time.time()
search.fit(X_search, y_search)
search_time = time.time() - start

print(f'Melhores hiperparametros: {search.best_params_}')
print(f'AP medio (CV) do melhor modelo: {search.best_score_:.4f}')
print(f'Tempo da busca: {search_time/60:.1f} min')

Busca em subamostra: 300,000 de 1,240,487 amostras de treino
Fitting 3 folds for each of 12 candidates, totalling 36 fits


[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   3.6s


[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   3.5s


[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   3.6s


[CV] END max_depth=None, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   3.5s


...

[CV] END max_depth=30, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   3.5s


[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   3.0s


[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   3.0s


[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   3.0s


Melhores hiperparametros: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}
AP medio (CV) do melhor modelo: 0.9601
Tempo da busca: 3.4 min


## 5. Treino do modelo final

Re-treina o melhor estimador encontrado na busca sobre todo o conjunto de
treino, medindo o tempo de treino isoladamente (para comparar custo
computacional entre os tres modelos no artigo).

In [5]:
best_rf = RandomForestClassifier(
    **search.best_params_,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

start = time.time()
best_rf.fit(X_train, y_train)
train_time = time.time() - start
print(f'Tempo de treino do modelo final: {train_time:.1f}s')

Tempo de treino do modelo final: 46.1s


## 6. Avaliacao

Mesmas metricas usadas no LSTM e na Regressao Logistica - acuracia, precisao,
recall, F1-score e matriz de confusao - calculadas em teste e validacao.

In [6]:
test_proba = best_rf.predict_proba(X_test)[:, 1]
val_proba  = best_rf.predict_proba(X_validation)[:, 1]

thr = ssd.best_threshold_f1(y_validation, val_proba)   # escolhido na validacao
print(f'Limiar otimo (F1): {thr:.4f}')

test_pred = (test_proba >= thr).astype(int)
val_pred  = (val_proba  >= thr).astype(int)

test_metrics = ssd.evaluate_predictions(y_test, test_pred)
val_metrics  = ssd.evaluate_predictions(y_validation, val_pred)
ssd.print_metrics('teste', test_metrics); print()
ssd.print_metrics('validacao', val_metrics)

Limiar otimo (F1): 0.3544


--- TESTE ---
Accuracy:  0.8779
Precision: 0.3353
Recall:    0.3650
F1-score:  0.3495
Matriz de confusão ([[TN, FP], [FN, TP]]):
[[224147  17253]
 [ 15142   8703]]

--- VALIDACAO ---
Accuracy:  0.8745
Precision: 0.3244
Recall:    0.3745
F1-score:  0.3477
Matriz de confusão ([[TN, FP], [FN, TP]]):
[[224774  18612]
 [ 14930   8938]]


## 7. Importancia das features

Vantagem do Random Forest sobre a Regressao Logistica e o LSTM: ele expoe
diretamente quais atributos SMART (e quais estatisticas de janela - atual,
media, desvio, min, max) mais contribuem para prever a falha. Isso e material
direto para a secao de discussao/resultados do artigo.

In [7]:
import pandas as pd

importances = pd.Series(best_rf.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)

print('Top 15 features mais importantes:')
print(importances.head(15).to_string())

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
top_n = 15
importances.head(top_n)[::-1].plot(kind='barh', ax=ax, color='#2E7D32', edgecolor='black', linewidth=0.5)
ax.set_title(f'Random Forest - Top {top_n} Features mais Importantes')
ax.set_xlabel('Importancia (reducao media de impureza)')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'rf_feature_importance_30d.png'), dpi=130, bbox_inches='tight')
print('Grafico salvo em rf_feature_importance_30d.png')

Top 15 features mais importantes:
smart_05_std      0.056243
smart_23_std      0.023856
smart_05_media    0.023531
smart_22_min      0.023449
smart_05_min      0.023037
smart_05_atual    0.022804
smart_05_max      0.022645
smart_00_max      0.022504
smart_23_atual    0.022296
smart_22_atual    0.022231
smart_23_min      0.022138
smart_00_min      0.022068
smart_23_media    0.022045
smart_23_max      0.022000
smart_22_media    0.021990


Grafico salvo em rf_feature_importance_30d.png


## 8. Salvando resultados para comparacao posterior

Mesmo formato de `resultados_logistic_regression.pkl`, com a adicao das
importancias de features (exclusivas do Random Forest). Isso permite montar
os graficos comparativos finais (ROC, precisao-recall, barras de metricas,
tempo de treino) entre os tres modelos para a apresentacao, sem precisar
re-rodar nada.

In [8]:
results = {
    'model_name': 'Random Forest',
    'contamination_level': CONTAMINATION_LEVEL,
    'window': WINDOW,
    'feature_cols': feature_cols,
    'feature_importances': importances.to_dict(),
    'best_params': search.best_params_,
    'cv_best_ap': search.best_score_,
    'search_time_seconds': search_time,
    'train_time_seconds': train_time,
    'threshold': thr,
    'metrics': {'test': test_metrics, 'validation': val_metrics},
    'predictions': {
        'test': {
            'disco_id': test['disco_id'], 'timestep': test['timestep'],
            'y_true': y_test, 'y_pred': test_pred, 'y_proba': test_proba,
        },
        'validation': {
            'disco_id': validation['disco_id'], 'timestep': validation['timestep'],
            'y_true': y_validation, 'y_pred': val_pred, 'y_proba': val_proba,
        },
    },
}

ssd.save_results(os.path.join(OUTPUT_DIR, 'resultados_random_forest_30d.pkl'), **results)

with open(os.path.join(OUTPUT_DIR, 'modelo_random_forest_30d.pkl'), 'wb') as f:
    pickle.dump({'model': best_rf}, f)
print('Modelo salvo em modelo_random_forest_30d.pkl')

Resultados salvos em: C:\Users\len108\Documents\pv\ErroSSD\horizonte_30dias\resultados_random_forest_30d.pkl


Modelo salvo em modelo_random_forest_30d.pkl


## Curva de aprendizagem

Desempenho (**Average Precision**) em funcao do tamanho do conjunto de treino.
Para cada fracao (de 2% ate **100%** do treino) o modelo e re-treinado e avaliado
no **mesmo split de validacao do projeto** (o split por disco usado no numero
final) - entao o **ultimo ponto coincide com o AP do modelo final** mostrado em
`AMA_comparacao_modelos`. A linha tracejada e o AP medido no proprio treino
(diagnostico de overfitting); a linha cheia e a validacao. E um diagnostico extra
e **nao** altera o modelo salvo nem os resultados.

> Re-treina o modelo em fracoes crescentes (incl. 100% = treino completo), entao
> custa ~2x um treino. Para acelerar, reduza a lista `fracs`. Os hiperparametros
> sao os ja escolhidos na busca (lidos da memoria ou do `.pkl` de resultados).

In [9]:
# Curva de aprendizagem FIEL: AP x tamanho do treino, avaliando no split de validacao do projeto.
import os, pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# hiperparametros: usa os da busca (se na memoria) ou le do .pkl de resultados
try:
    _bp = dict(search.best_params_)
except NameError:
    with open(os.path.join(OUTPUT_DIR, 'resultados_random_forest_30d.pkl'), 'rb') as _f:
        _bp = pickle.load(_f)['best_params']

base_est = RandomForestClassifier(**_bp, class_weight='balanced',
                                  random_state=RANDOM_STATE, n_jobs=-1)

Xtr_full, ytr_full = train['X'], train['y']     # treino completo (sem escalonar; pipeline cuida disso)
Xval, yval = validation['X'], validation['y']   # split de validacao do projeto (mesmo do numero final)

fracs = [0.02, 0.05, 0.1, 0.25, 0.5, 1.0]
sizes, ap_tr, ap_va = [], [], []
print('Curva de aprendizagem (avaliando na validacao do projeto):')
for fr in fracs:
    if fr < 1.0:
        X_s, _, y_s, _ = train_test_split(Xtr_full, ytr_full, train_size=fr,
                                          stratify=ytr_full, random_state=RANDOM_STATE)
    else:
        X_s, y_s = Xtr_full, ytr_full
    est = clone(base_est)
    est.fit(X_s, y_s)
    ap_v = average_precision_score(yval, est.predict_proba(Xval)[:, 1])
    ap_t = average_precision_score(y_s, est.predict_proba(X_s)[:, 1])
    sizes.append(len(y_s)); ap_va.append(ap_v); ap_tr.append(ap_t)
    print(f'  treino={len(y_s):>9,}  AP_treino={ap_t:.4f}  AP_val={ap_v:.4f}')

sizes = np.array(sizes); ap_tr = np.array(ap_tr); ap_va = np.array(ap_va)
_prev = float(np.mean(yval))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(sizes, ap_tr, 'o--', color='#1565C0', label='Treino (no proprio subconjunto)')
ax.plot(sizes, ap_va, 'o-', color='#E65100', label='Validacao (split do projeto)')
ax.axhline(_prev, color='gray', ls=':', lw=1, label=f'Baseline aleatorio (prevalencia={_prev:.3f})')
ax.set_xlabel('Amostras de treino')
ax.set_ylabel('Average Precision (AP)')
ax.set_title('Curva de aprendizagem - Random Forest (30 dias)')
ax.legend(loc='best')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'rf_learning_curve_30d.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Curva de aprendizagem salva em rf_learning_curve_30d.png  (ultimo ponto = AP do modelo final na validacao)')

# Salva os dados para a figura consolidada da comparacao (std=0: estimativa unica por ponto)
with open(os.path.join(OUTPUT_DIR, 'rf_learning_curve_30d.pkl'), 'wb') as _f:
    pickle.dump({'model_name': 'Random Forest', 'scoring': 'average_precision',
                 'train_sizes': sizes,
                 'train_mean': ap_tr, 'train_std': np.zeros_like(ap_tr),
                 'val_mean': ap_va, 'val_std': np.zeros_like(ap_va)}, _f)
print('Dados da curva salvos em rf_learning_curve_30d.pkl')


Curva de aprendizagem (avaliando na validacao do projeto):


  treino=   24,809  AP_treino=1.0000  AP_val=0.2745


  treino=   62,024  AP_treino=1.0000  AP_val=0.2999


  treino=  124,048  AP_treino=1.0000  AP_val=0.3019


  treino=  310,121  AP_treino=1.0000  AP_val=0.3062


  treino=  620,243  AP_treino=1.0000  AP_val=0.3026


  treino=1,240,487  AP_treino=1.0000  AP_val=0.3051
Curva de aprendizagem salva em rf_learning_curve_30d.png  (ultimo ponto = AP do modelo final na validacao)
Dados da curva salvos em rf_learning_curve_30d.pkl


C:\Users\len108\AppData\Local\Temp\ipykernel_18428\2062486190.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
